Решить задачу машинного перевода выбрав свой язык:
+ Формируем датасет с исходного языка на целевой (код прописать в классе)
+ Строим архитектуру нейронной сети 
+ Обучаем 
+ Проверить качество с помощью метрики BLEU

### По стандарту импортируем все необходимые модули и библиотки для решения задачи Seq2Seq для задачи машинного перевода:

In [1]:
from io import open
import unicodedata
import string
import re
import random
import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F
import warnings

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#### Постановка задачи:
----

Задача - научить RNNs переводить предложения с одного языка на другой язык. Используем файл rus.txt для того чтобы научить модель переводить с RU на EN язык.
Структура файла - обычный текстовой файл, состоящий из строк, где каждая строка 1 предложение на исходном языке (EN) Затем знак табуляции \t и предложение на целевом языке и так несколько строк в файле.
Нам необходимо сформировать искусственно класс словаря, провести кодировку последовательности и сформировать датасет.

## 1. Подготовим данные:

In [2]:
# Определим по умолчанию 2 токена которые будут нам информировать о начале предложения и конце предложения (SOS и EOS):
SOS_token = 0
EOS_token = 1

# Создадим объект словаря нашего языка, который будет хранить данные по маппингу слов - index2word и обратно word2index и плюс второстепенные методы по добавлению токена и обработке предложений:
class LanguageVocabulary(object):
    def __init__(self, name):
        # название языка
        self.name = name
        # словарик word2index который хранит соответственно кодировку слова в целочисленный индекс словаря
        self.word2index = {}
        # обычный словарик который хранит распределение слов, сколько слов мы использовали и сколько обнаружили
        self.word2count = {}
        # Обратный словарик словарю word2index где хранятся уже индексы и замаппенные слова к каждому индексу, нужен будет для расшифровки последовательности
        self.index2word = {0: "SOS", 1: "EOS"}
        # Count SOS and EOS, храним просто общее количество слов в нашем словаре, то есть количество токенов в сформированном словарике нашего языка
        self.n_words = 2

    def add_sentence(self, sentence):
        """
        Метод класса, для добавления предложения в словарь.
        Каждое предложение поступающее к нам, будет разбираться на
        примитивные токены и добавляться в словарь при помощи метода класса addword()
        """
        for word in sentence.split(' '):
            self.add_word(word)


    def add_word(self, word):
        # проверяем не входит ли наше слово в словарь word2index
        if word not in self.word2index:
            # добавляем в качестве ключа слово а в качестве значения последнее n_words
            self.word2index[word] = self.n_words
            # меняем на единичку
            self.word2count[word] = 1
            # и соответственно меняем и index2word словарик добавляя уже слово для декодирования
            self.index2word[self.n_words] = word
            # инкрементируем n_words
            self.n_words += 1
        else:
            # Если такое уже слово есть просто добавляем 1 что добавилось одно слово
            self.word2count[word] += 1

##### Класс словарика построили, теперь надо подумать как обработать последовательность текста. Создадим 2 фукнции uncode_to_ascii() и normalize_string(). Как мы помним из специфики работы Python, питоновский интерпритатор представляет строку в виде юникод кодировки, что неудобно. Необходимо перевести все в кодировку стандарта ASCII (FYI для разных языков, разный подход, прежде чем приступать к раскодировке из одного языка в другой, поищите информацию о представлении языка в памяти компьютера, например для Китайского, Японского, Иврита, Арабского и прочих языков есть своя определенная специфика). Обычно все насущные вопросы уже кем-то были решены и все решается просто поиском ответа на stackoverflow как я и сделал ниже:

In [3]:

def unicode_to_ascii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

# Lowercase, trim, and remove non-letter characters (but keep Cyrillic)
def normalize_string(s):
    # Декодируем из юникода в ascii (для латиницы)
    s = unicode_to_ascii(s.lower().strip())
    # Знаки препинания отделяем пробелами
    s = re.sub(r"([.!?])", r" \1", s)
    # Оставляем латиницу, кириллицу, точки, восклицательные и вопросительные знаки
    s = re.sub(r"[^a-zA-Zа-яА-Я.!?]+", r" ", s)
    return s

##### Создадим функцию которая будет просто открывать наши документы корпуса и создавать словарик:

In [4]:
def read_languages(lang1, lang2, reverse=False):
    print("Reading lines...")
    file_path = r"C:\Users\user\Downloads\rus-eng\rus.txt"
    with open(file_path, encoding='utf-8') as f:
        lines = f.read().strip().split('\n')
    lines = [l for l in lines if '\t' in l]
    pairs = []
    for line in lines:
        parts = line.split('\t')
        if len(parts) < 2:
            continue
        s1 = normalize_string(parts[0])
        s2 = normalize_string(parts[1])
        # дополнительная проверка: если после нормализации строка пуста, пропускаем
        if not s1.strip() or not s2.strip():
            continue
        pairs.append([s1, s2])
    
    if reverse:
        pairs = [list(reversed(p)) for p in pairs]
        input_lang = LanguageVocabulary(lang2)
        output_lang = LanguageVocabulary(lang1)
    else:
        input_lang = LanguageVocabulary(lang1)
        output_lang = LanguageVocabulary(lang2)

    return input_lang, output_lang, pairs

##### Чтобы немного упростить задачу и соответственно время на ее решение, для упрощения мы просто возьмем и отберем часть предложений которые будут начинаться на биграммы указанные в eng_prefixes и соответсвенно ограничим длину предложения документа 10-ю символами:

In [5]:
MAX_LENGTH = 10

#Убираем eng_prefixes, так как для русско-английского датасета они не нужны
def filter_pair(p):
    return len(p[0].split(' ')) < MAX_LENGTH and len(p[1].split(' ')) < MAX_LENGTH

def filter_pairs(pairs):
    return [pair for pair in pairs if filter_pair(pair)]



#### Создадим функцию которая будет возвращать нам уже данные:

In [6]:
def prepare_data(lang1, lang2, reverse=False):
    input_lang, output_lang, pairs = read_languages(lang1, lang2, reverse)
    print("Read %s sentence pairs" % len(pairs))
    pairs = filter_pairs(pairs)
    print("Trimmed to %s sentence pairs" % len(pairs))
    print("Counting words...")
    for pair in pairs:
        input_lang.add_sentence(pair[0])
        output_lang.add_sentence(pair[1])
    print("Counted words:")
    print(input_lang.name, input_lang.n_words)
    print(output_lang.name, output_lang.n_words)
    return input_lang, output_lang, pairs

In [7]:
input_lang, output_lang, pairs = prepare_data('eng', 'ru', True)
print(random.choice(pairs))

Reading lines...
Read 536124 sentence pairs
Trimmed to 441422 sentence pairs
Counting words...
Counted words:
ru 52680
eng 15824
['почему вы мне об этом не сказали ?', 'why didn t you tell me ?']


In [8]:
pairs

[['марш !', 'go .'],
 ['иди .', 'go .'],
 ['идите .', 'go .'],
 ['здравствуите .', 'hi .'],
 ['привет !', 'hi .'],
 ['хаи .', 'hi .'],
 ['здрасте .', 'hi .'],
 ['здорово !', 'hi .'],
 ['приветик !', 'hi .'],
 ['беги !', 'run !'],
 ['бегите !', 'run !'],
 ['беги !', 'run .'],
 ['бегите !', 'run .'],
 ['бегите .', 'run .'],
 ['беги .', 'run .'],
 ['кто ?', 'who ?'],
 ['вот это да !', 'wow !'],
 ['круто !', 'wow !'],
 ['здорово !', 'wow !'],
 ['ух ты !', 'wow !'],
 ['ого !', 'wow !'],
 ['вах !', 'wow !'],
 ['вау !', 'wow !'],
 ['пригнись !', 'duck !'],
 ['огонь !', 'fire !'],
 ['пожар !', 'fire !'],
 ['помогите !', 'help !'],
 ['на помощь !', 'help !'],
 ['спасите !', 'help !'],
 ['помоги !', 'help !'],
 ['прячься .', 'hide .'],
 ['прячьтесь .', 'hide .'],
 ['спрячься .', 'hide .'],
 ['спрячьтесь .', 'hide .'],
 ['прыгаи !', 'jump !'],
 ['прыгаите !', 'jump !'],
 ['прыгаи !', 'jump .'],
 ['прыгаите !', 'jump .'],
 ['оставаися .', 'stay .'],
 ['оставаитесь .', 'stay .'],
 ['останься .', 's

# 2. Построим  Encoder
-----------


In [9]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(EncoderRNN, self).__init__()
        # Как помним hidden_size - размер скрытого состояния
        self.hidden_size = hidden_size
        # Слой Эмбеддингов, который из входного вектора последовательности (либо батча) отдаст представление последовательности для скрытого состояния
        # FYI: в качестве Input_size у нас размер словаря
        self.embedding = nn.Embedding(input_size, hidden_size)
        # И соответственно рекуррентная ячейка GRU которая принимает MxM (hidden на hidden)
        self.gru = nn.GRU(hidden_size, hidden_size)

    def forward(self, input, hidden):
        # Приводим эмбеддинг к формату одного предлоежния 1х1 и любая размерность
        embedded = self.embedding(input).view(1, 1, -1)
        # Нужно для следующего шага пока не запутываемся :) просто присвоили наш эмбеддинг
        output = embedded
        # и соответственно подаем все в ГРЮ ячейку (эмбеддинг и скрытые состояния)
        output, hidden = self.gru(output, hidden)
        return output, hidden

    def initHidden(self):
        # Дополнительно сделаем инициализацию скрытого представления (просто заполним нулями)
        return torch.zeros(1, 1, self.hidden_size, device=device)

# 3. Построим  Decoder
-----------


Механизм Attention в данной реализации используется для улучшения качества перевода или генерации последовательностей в задачи, связанные с машинным переводом или с другими задачами последовательной обработки данных. Рассмотрим, как этот механизм реализован в классе AttentionDecoder.
1. Инициализация компонентов:

- embedding: Это слой для преобразования входных токенов в их скрытые представления (embedding). Токены (например, слова) из выходной последовательности преобразуются в векторы фиксированного размера (равного hidden_size).

- attn: Это линейный слой, который используется для вычисления весов внимания. Он принимает скрытое состояние декодера и эмбеддинг текущего входа, чтобы вычислить важность каждого скрытого состояния в выходной последовательности для текущего шага.

- attn_combine: Этот слой комбинирует входное внимание и результат, полученный от слоя внимания. Он создает вектор, который будет передан в GRU.

- dropout: Слой Dropout используется для регуляризации, чтобы избежать переобучения модели.

- gru: Это основной рекуррентный слой, который обновляет скрытое состояние декодера на каждом шаге. Он использует входное скрытое состояние и выход на предыдущем шаге, чтобы вычислить текущее скрытое состояние.

- out: Линейный слой для преобразования скрытого состояния в выходной токен, соответствующий словарю.

2. Процесс Attention в forward:

Вводные данные:
- input: Это текущий входной токен, который декодер должен предсказать.
- hidden: Это скрытое состояние декодера на текущем шаге.
- encoder_outputs: Это все скрытые состояния энкодера, которые содержат информацию о всей исходной последовательности.

Эмбеддинг и Dropout:
- Входной токен преобразуется в скрытое представление через слой embedding.
- Далее на эмбеддинг накладывается слой dropout, чтобы предотвратить переобучение.

Вычисление весов внимания:
- Веса внимания (attn_weights) вычисляются через слой attn, который принимает конкатенированные скрытые состояния текущего шага декодера и эмбеддинг входного токена.
  - Применяется функция активации Softmax для нормализации весов, чтобы они суммировались в 1 и могли интерпретироваться как вероятности.

Применение внимания:
- После того как веса внимания вычислены, они умножаются на скрытые состояния энкодера через операцию batch matrix multiplication (torch.bmm). Это дает результат, который представляет собой контекстное внимание, учитывающее скрытые состояния, относящиеся к наиболее важным частям входной последовательности для текущего шага декодера.

Конкатенация с эмбеддингом:
- Результат применения внимания (attn_applied) конкатенируется с эмбеддингом текущего токена, чтобы получить объединенную информацию о текущем шаге и контексте.

Процесс через GRU:
- Эта объединенная информация передается в слой GRU, который обновляет скрытое состояние и генерирует выходной вектор для текущего шага декодера.

Выход:
- Скрытое состояние декодера преобразуется через линейный слой out, и результат подвергается логарифмическому Softmax, чтобы получить вероятности для всех возможных выходных токенов.

3. Результаты:

- output: Это распределение вероятностей для следующего токена в последовательности.
- hidden: Это обновленное скрытое состояние, которое будет передано на следующий шаг.
- attn_weights: Это веса внимания, которые показывают, какие части исходной последовательности были наиболее важны для предсказания текущего токена.

Преимущества механизма внимания:

- Контекстность: Механизм внимания позволяет декодеру выбирать, какие части входной последовательности наиболее релевантны для предсказания каждого токена, что помогает модели обрабатывать длинные последовательности и захватывать важные зависимости между токенами.
- Гибкость: Механизм внимания помогает справляться с различными длинами последовательностей, что особенно важно в задачах машинного перевода, где длина входной и выходной последовательности может сильно различаться.

В итоге, этот механизм позволяет декодеру использовать информацию о всей входной последовательности для улучшения точности предсказания на каждом шаге, обеспечивая динамическое внимание к наиболее важным частям входного текста.

In [10]:
class AttentionDecoder(nn.Module):
    # Будьте внимательны, теперь на вход мы получаем размер скрытого представления
    def __init__(self, hidden_size, output_size, dropout_p=0.1, max_length=MAX_LENGTH):
        super(AttentionDecoder, self).__init__()
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.dropout_p = dropout_p
        self.max_length = max_length

        self.embedding = nn.Embedding(self.output_size, self.hidden_size)
        self.attn = nn.Linear(self.hidden_size * 2, self.max_length)
        self.attn_combine = nn.Linear(self.hidden_size * 2, self.hidden_size)
        self.dropout = nn.Dropout(self.dropout_p)
        self.gru = nn.GRU(self.hidden_size, self.hidden_size)
        self.out = nn.Linear(self.hidden_size, self.output_size)

    def forward(self, input, hidden, encoder_outputs):
        embedded = self.embedding(input).view(1, 1, -1)
        embedded = self.dropout(embedded)

        attn_weights = F.softmax(
            self.attn(torch.cat((embedded[0], hidden[0]), 1)), dim=1)
        attn_applied = torch.bmm(attn_weights.unsqueeze(0),
                                 encoder_outputs.unsqueeze(0))

        output = torch.cat((embedded[0], attn_applied[0]), 1)
        output = self.attn_combine(output).unsqueeze(0)

        output = F.relu(output)
        output, hidden = self.gru(output, hidden)

        output = F.log_softmax(self.out(output[0]), dim=1)
        return output, hidden, attn_weights

    def initHidden(self):
        return torch.zeros(1, 1, self.hidden_size, device=device)

# 4. Создадим вспомогательные функции для работы с полученной репрезентацией и для кодирования для подачи в модель:

In [11]:
# Токены кодируем в целочисленное представление
def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(' ')]


# Берем предложение с указанным языком, делаем из него индексы и вставляем метку конца предложения, превращаем в тензор:
def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(-1, 1)

# Для создания тензора из пар:
def tensorsFromPair(pair):
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return (input_tensor, target_tensor)

# 5. Создадим функцию обучения для работы только с ОДНОЙ парой:

Teacher Forcing — это техника, используемая в задачах с последовательной генерацией данных (например, машинный перевод, генерация текста), которая помогает ускорить обучение модели. Суть метода заключается в том, что на каждом шаге декодирования, в качестве входа в декодер используется правильный токен из целевой последовательности (а не предсказанный моделью токен). Это помогает модели быстрее научиться предсказывать правильную последовательность, потому что она всегда получает правильный контекст.

В коде параметр teacher_forcing_ratio управляет частотой применения teacher forcing. Если случайное число меньше этого значения, то на текущем шаге обучения используется настоящий токен из целевой последовательности, а не токен, предсказанный моделью.

In [12]:
teacher_forcing_ratio = 0.5


def train(input_tensor, target_tensor, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion, max_length=MAX_LENGTH):
    # Просто инициализируем скрытое представление для энкодера
    encoder_hidden = encoder.initHidden()
    # Скиыдваем градиенты для алгоритма градиентного спуска как и у энкодера так и у дэкодера
    encoder_optimizer.zero_grad()
    decoder_optimizer.zero_grad()
    # Получаем размер в словаря (токенов) для входящего и выходящего тензора так как мы пробегаемся по каждому предложению по кусочкам
    input_length = input_tensor.size(0)
    target_length = target_tensor.size(0)
    # Создаем переменную где будем хранить наши выходы из энкодера (в данной реализации пока не юзаем, далее будет еще один вариант)
    encoder_outputs = torch.zeros(max_length, encoder.hidden_size, device=device)
    loss = 0
    # пробегаем по длине входящего тензора и в экодер передаем последовательно каждый из токенов:
    for ei in range(input_length):
        encoder_output, encoder_hidden = encoder(input_tensor[ei], encoder_hidden)
        # Сохраняем все выходы из энкодера для одного слова (для передачи в декодер, ниже)
        encoder_outputs[ei] = encoder_output[0, 0]


    # Закончили с энкодером пошли к декодеру, как было сказано декодер начинается с SOS
    decoder_input = torch.tensor([[SOS_token]], device=device)
    # FYI здесь мы скрытое представление из энкодера передаем в скрытое представление в декодер, то есть после знака =
    # у нас будут ходить градиенты из декодера в энкодер, то есть когда мы будем считать градиенты, они сначала пробегут по декодеру
    # дойдут до знака = перескочат в энкодер и будут дальше считаться по энкодеру и эти градиенты сохранятся в соответствующих тензорах
    # и когда будут отрабатывать разные оптимайзеры (у нас их 2) у них будут соответствующие правильные градиенты которые смогут правильно отработать
    decoder_hidden = encoder_hidden

    # Будем использовать Teacher Forcing в части случае (подставляя правильную последовательность)
    use_teacher_forcing = True if random.random() < teacher_forcing_ratio else False
    if use_teacher_forcing:
        # Подаем decoder_input = torch.tensor([[SOS_token]], device=device) то есть по одному слову и скрытое представление
        for di in range(target_length):
            # Переведенное предложение и скрытое представление
            decoder_output, decoder_hidden, decoder_attention = decoder(decoder_input, decoder_hidden, encoder_outputs)
            # Считаем ошибку
            loss += criterion(decoder_output, target_tensor[di])
            decoder_input = target_tensor[di]  # Teacher forcing
    else:
        for di in range(target_length):
            decoder_output, decoder_hidden, decoder_attention = decoder(decoder_input, decoder_hidden, encoder_outputs)
            topv, topi = decoder_output.topk(1)
            decoder_input = topi.squeeze().detach()  # detach from history as input
            loss += criterion(decoder_output, target_tensor[di])
            if decoder_input.item() == EOS_token:
                break
    loss.backward()
    encoder_optimizer.step()
    decoder_optimizer.step()
    return loss.item() / target_length

# 6. Просто красивые функции для засекания времени:

In [13]:
import time
import math


def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)


def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / percent
    rs = es - s
    return '%s (- eta: %s)' % (asMinutes(s), asMinutes(rs))

# 7. Функция которая будет пробегаться по всем парам и использовать эти пары для обучения сети:

In [14]:
def trainIters(encoder, decoder, n_iters, print_every=1000, plot_every=100, learning_rate=0.01):
    start = time.time()
    plot_losses = []
    print_loss_total = 0  # Reset every print_every
    plot_loss_total = 0  # Reset every plot_every

    encoder_optimizer = optim.SGD(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.SGD(decoder.parameters(), lr=learning_rate)
    # Делаем выборку наших пар функцией которую создали до
    training_pairs = [tensorsFromPair(random.choice(pairs)) for i in range(n_iters)]
    # FYI! Используем Negative Log-Likelihood Loss потому что log softmax уже присутствует в модели
    criterion = nn.NLLLoss()

    for epoch in range(1, n_iters + 1):
        training_pair = training_pairs[epoch - 1]
        input_tensor = training_pair[0]
        target_tensor = training_pair[1]
        # Используем функцию для тренировки на отдельных токенах, которую написали выше
        loss = train(input_tensor, target_tensor, encoder,
                     decoder, encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss
        plot_loss_total += loss

        if epoch % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, epoch / n_iters),
                                         epoch, epoch / n_iters * 100, print_loss_avg))

        if epoch % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0
    showPlot(plot_losses)

In [15]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
plt.switch_backend('agg')


def showPlot(points):
    plt.figure()
    fig, ax = plt.subplots()
    loc = ticker.MultipleLocator(base=0.2)
    ax.yaxis.set_major_locator(loc)
    plt.plot(points)

# 8. Теперь построим функцию которая позволит нам использовать Encoder-Decoder для перевода предложения:

In [16]:
def evaluate_sentences(encoder, decoder, sentence, max_length=MAX_LENGTH):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)
        input_length = input_tensor.size()[0]
        encoder_hidden = encoder.initHidden()

        encoder_outputs = torch.zeros(max_length, encoder.hidden_size, device=device)

        for ei in range(input_length):
            encoder_output, encoder_hidden = encoder(input_tensor[ei],
                                                     encoder_hidden)
            encoder_outputs[ei] += encoder_output[0, 0]

        decoder_input = torch.tensor([[SOS_token]], device=device)  # SOS

        decoder_hidden = encoder_hidden

        decoded_words = []
        decoder_attentions = torch.zeros(max_length, max_length)

        for di in range(max_length):
            decoder_output, decoder_hidden, decoder_attention = decoder(
                decoder_input, decoder_hidden, encoder_outputs)
            decoder_attentions[di] = decoder_attention.data
            topv, topi = decoder_output.data.topk(1)
            if topi.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            else:
                decoded_words.append(output_lang.index2word[topi.item()])

            decoder_input = topi.squeeze().detach()

        return decoded_words, decoder_attentions[:di + 1]

In [17]:
import evaluate

bleu = evaluate.load("bleu")

references, predictions = [],[]
def evaluateRandomly(encoder, decoder, n=10):
    for i in range(n):
        pair = random.choice(pairs)
        print('>', pair[0])
        print('=', pair[1])
        references.append(pair[1])
        output_words, attentions = evaluate_sentences(encoder, decoder, pair[0])
        output_sentence = ' '.join(output_words)
        print('<', output_sentence)
        predictions.append(output_sentence)
        print('')

    results = bleu.compute(predictions=predictions, references=references)
    return f"BLEU Score: {results}"

# 9. Этап обучения:

In [18]:
evaluate.list_evaluation_modules()

['lvwerra/test',
 'angelina-wang/directional_bias_amplification',
 'cpllab/syntaxgym',
 'lvwerra/bary_score',
 'hack/test_metric',
 'yzha/ctc_eval',
 'codeparrot/apps_metric',
 'mfumanelli/geometric_mean',
 'daiyizheng/valid',
 'erntkn/dice_coefficient',
 'mgfrantz/roc_auc_macro',
 'Vlasta/pr_auc',
 'gorkaartola/metric_for_tp_fp_samples',
 'idsedykh/metric',
 'idsedykh/codebleu2',
 'idsedykh/codebleu',
 'idsedykh/megaglue',
 'Vertaix/vendiscore',
 'GMFTBY/dailydialogevaluate',
 'GMFTBY/dailydialog_evaluate',
 'jzm-mailchimp/joshs_second_test_metric',
 'ola13/precision_at_k',
 'yulong-me/yl_metric',
 'abidlabs/mean_iou',
 'abidlabs/mean_iou2',
 'KevinSpaghetti/accuracyk',
 'NimaBoscarino/weat',
 'ronaldahmed/nwentfaithfulness',
 'Viona/infolm',
 'kyokote/my_metric2',
 'kashif/mape',
 'Ochiroo/rouge_mn',
 'leslyarun/fbeta_score',
 'anz2/iliauniiccocrevaluation',
 'zbeloki/m2',
 'xu1998hz/sescore',
 'dvitel/codebleu',
 'NCSOFT/harim_plus',
 'JP-SystemsX/nDCG',
 'sportlosos/sescore',
 'Dru

In [19]:
hidden_size = 512
encoder1 = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder1 = AttentionDecoder(hidden_size, output_lang.n_words, dropout_p=0.1).to(device)
trainIters(encoder1, decoder1, 75000, print_every=5000)

38m 36s (- eta: 540m 30s) (5000 6%) 4.3686
77m 51s (- eta: 506m 2s) (10000 13%) 3.7073
116m 54s (- eta: 467m 36s) (15000 20%) 3.4180
157m 14s (- eta: 432m 23s) (20000 26%) 3.2239
197m 55s (- eta: 395m 50s) (25000 33%) 3.0672
238m 52s (- eta: 358m 18s) (30000 40%) 2.9723
279m 28s (- eta: 319m 23s) (35000 46%) 2.8639
317m 50s (- eta: 278m 6s) (40000 53%) 2.7878
356m 28s (- eta: 237m 39s) (45000 60%) 2.7385
395m 22s (- eta: 197m 41s) (50000 66%) 2.6673
434m 10s (- eta: 157m 53s) (55000 73%) 2.5938
473m 14s (- eta: 118m 18s) (60000 80%) 2.5796
512m 14s (- eta: 78m 48s) (65000 86%) 2.5349
551m 18s (- eta: 39m 22s) (70000 93%) 2.4881
590m 28s (- eta: 0m 0s) (75000 100%) 2.4683


# 10. Тестируем результат:

In [20]:
score = evaluateRandomly(encoder1, decoder1, n=1000)

> некоторые говорят что я непредсказуема .
= some people say i m unpredictable .
< some people i i m . . <EOS>

> это кольцо мне подарила мать .
= my mother gave me this ring .
< this is a my to to . . <EOS>

> можешь спеть песню .
= you can sing a song .
< you can sing a song . <EOS>

> почему ты уволилась с работы ?
= why did you quit your job ?
< why are you looking for my job ? <EOS>

> могу я воспользоваться сегодня твоим автомобилем ?
= may i use your car today ?
< may i use today your phone ? <EOS>

> жаль что мы не можем поехать в бостон .
= i wish we could go to boston .
< i wish we we go go boston . <EOS>

> все что мы тебе сказали правда .
= everything we ve told you is true .
< everyone told you all tell . . <EOS>

> что вам не нравится в вашеи работе ?
= what don t you like about your job ?
< what don t you like like your work <EOS>

> я не могу туда поити .
= i can t go there .
< i can t go there . <EOS>

> я сделал только три ошибки .
= i only made three mistakes .
< i o

In [21]:
score

"BLEU Score: {'bleu': 0.12969599800273965, 'precisions': [0.408627284317892, 0.17772230147408463, 0.08796546141392336, 0.044291952588895823], 'brevity_penalty': 1.0, 'length_ratio': 1.406035255452644, 'translation_length': 9412, 'reference_length': 6694}"